# NN and TNN models: Figures 1-4

MoS$_2$ (GGA), NN and TNN models, at $B = 0$ and at a large field $\mu_BB = 80$ meV ($B \approx 1.4\times10^3$ T) so the splitting is visible. Only the Zeeman Hamiltonian enters here, no moment shift. About a minute.

## Load engine

In [ ]:
%run engine.ipynb

## Settings

In [ ]:
N_geo = 600        # grid for the maps of Figs. 2-4 (200 for a quick look)

cases = [("NN", 0), ("TNN", 0), ("NN", B80), ("TNN", B80)]
p_NN, p_TNN = params("MoS2", model="NN"), params("MoS2", model="TNN")
lam = lambda_SOC["MoS2"]

## Figure 1: bands along $\Gamma$-$K$-$M$-$\Gamma$

Spin up blue, spin down red, energies measured from the top of the valence band. The $B = 0$ bands are drawn in grey under the Zeeman panels.

In [ ]:
u = np.linspace(0, 1, 200)          # goes from 0 to 1 along each piece of the path
K_pt = np.array([K_len, 0])
M_pt = np.array([np.pi, np.pi / np.sqrt(3)])
k_path = np.concatenate([np.outer(u, K_pt),                          # Gamma -> K
                         K_pt + np.outer(u, M_pt - K_pt),            # K -> M
                         M_pt - np.outer(u, M_pt)])                  # M -> Gamma
x_path = np.concatenate([[0], np.cumsum(np.linalg.norm(np.diff(k_path, axis=0), axis=1))])     # distance along the path
x_corners = x_path[[0, 200, 400, 599]]

path_bands = {}
for model, B in cases:
    p = p_NN if model == "NN" else p_TNN
    H_up, H_dn = zeeman(hamiltonian(model, p, k_path[:, 0], k_path[:, 1]), lam, B)
    E_up, E_dn = np.linalg.eigvalsh(H_up), np.linalg.eigvalsh(H_dn)
    top = max(E_up[:, 0].max(), E_dn[:, 0].max())
    path_bands[model, B] = (E_up - top, E_dn - top)

H_up, H_dn = zeeman(hamiltonian("TNN", p_TNN, np.array([K_len]), np.array([0.0])), lam, 0)
E_up, E_dn = np.linalg.eigvalsh(H_up)[0], np.linalg.eigvalsh(H_dn)[0]
print(f"TNN model: gap at K = {E_up[1] - E_up[0]:.4f} eV, valence spin splitting = {E_up[0] - E_dn[0]:.4f} eV")

In [ ]:
big_style = {"font.family": "DejaVu Serif", "mathtext.fontset": "cm", "font.size": 13, "axes.linewidth": 1.0, "savefig.bbox": "standard",
             "xtick.labelsize": 13, "ytick.labelsize": 13, "xtick.direction": "out", "ytick.direction": "out",
             "xtick.major.size": 3.5, "ytick.major.size": 3.5, "xtick.major.width": 1.0, "ytick.major.width": 1.0,
             "xtick.minor.visible": False, "ytick.minor.visible": False, "xtick.top": False, "ytick.right": False}


def panel(fig, x, top, width, height):
    # axes from their position on the page, in points from the top left corner
    W, H = fig.get_size_inches() * 72
    return fig.add_axes([x / W, 1 - (top + height) / H, width / W, height / H])


with plt.rc_context(big_style):
    fig = plt.figure(figsize=(1006.25 / 72, 271.24 / 72))
    for j, (model, B) in enumerate(cases):
        ax = panel(fig, 46.8 + 238.1 * j, 24.8, 238.1, 224.4)
        E_up, E_dn = path_bands[model, B]
        if B != 0:
            ax.plot(x_path, path_bands[model, 0][0], color="0.7", lw=1.2, zorder=1)
            ax.plot(x_path, path_bands[model, 0][1], color="0.7", lw=1.2, zorder=1)
        ax.plot(x_path, E_dn, color="#E31A1C", lw=2, ls=(0, (3.7, 1.6)), zorder=2)
        ax.plot(x_path, E_up, color="#2060C8", lw=2, zorder=3)
        ax.set_xlim(x_path.min(), x_path.max())
        ax.set_ylim(-1.35, 4.05)
        ax.set_xticks(x_corners, [r"$\Gamma$", "K", "M", r"$\Gamma$"])
        ax.set_yticks(range(-1, 5))
        ax.tick_params(labelleft=(j == 0), left=(j == 0))
        ax.grid(color="lightgray", lw=1)
        ax.set_axisbelow(True)
        ax.text(0.046, 0.929, "(" + "abcd"[j] + ")", transform=ax.transAxes, fontsize=18, fontweight="bold", va="top")
        ax.text(0.966, 0.963, model, transform=ax.transAxes, fontsize=12, fontweight="bold", ha="right", va="top")
    fig.text(9 / 1006.25, 0.5, "Energy (eV)", rotation=90, ha="center", va="center", fontsize=15)
    fig.text(285 / 1006.25, 1 - 8 / 271.24, "$B=0$", ha="center", va="center", fontsize=15)
    fig.text(761.5 / 1006.25, 1 - 8 / 271.24, "Zeeman field", ha="center", va="center", fontsize=15)
    plt.savefig("figures/bands_grid.pdf")
    plt.show()

## Berry curvature and quantum metric over the zone

The occupied bands are the valence bands of the two blocks, so the occupied Berry curvature is $\Omega = \Omega_{v\uparrow} + \Omega_{v\downarrow}$ and the metric is $g^{ab} = \sum_{v\,\in\,\rm occ}\sum_c g^{ab}_{cv}$, with $\Omega_n$ and $g_{cv}$ from the engine.
The Chern number of band $n$ is $\frac{1}{2\pi}\int_{\rm BZ}\Omega_n\,d^2k$.

In [ ]:
L = 1.16 * K_len
q = np.linspace(-L, L, N_geo)
k_xv, k_yv = np.meshgrid(q, q)
k_x, k_y, dA = slanted_grid(N_geo)

maps, chern = {}, {}
for model, B in cases:
    p = p_NN if model == "NN" else p_TNN
    Omega_occ, g_occ = 0, 0
    for s, spin in enumerate(["up", "dn"]):
        H = lambda x, y: zeeman(hamiltonian(model, p, x, y), lam, B)[s]
        E, V, v_x, v_y = bands(H, k_xv, k_yv)
        Omega, M = berry_and_moment(E, v_x, v_y, p["a"])
        Omega_occ = Omega_occ + Omega[..., 0]
        g_occ = g_occ + metric(E, v_x, v_y, 1) + metric(E, v_x, v_y, 2)
        if B == 0:
            E, V, v_x, v_y = bands(H, k_x, k_y)
            chern[model, spin] = np.sum(berry_and_moment(E, v_x, v_y, p["a"])[0], axis=0) * dA / (2 * np.pi)
    trace_g = g_occ[..., 0, 0] + g_occ[..., 1, 1]
    det_g = g_occ[..., 0, 0] * g_occ[..., 1, 1] - g_occ[..., 0, 1]**2
    maps[model, B] = [Omega_occ, trace_g, det_g]

## Chern numbers

In [ ]:
for model in ["NN", "TNN"]:
    for spin in ["up", "dn"]:
        C = chern[model, spin]
        print(f"{model}, spin {spin}: Chern numbers {C[0]:+.6f}, {C[1]:+.6f}, {C[2]:+.6f}")

## Figures 2-4: occupied Berry curvature, metric trace and metric determinant

Same four cases as Fig. 1, with the zone boundary drawn.

In [ ]:
hex_x = K_len * np.cos(np.arange(7) * np.pi / 3)      # zone boundary
hex_y = K_len * np.sin(np.arange(7) * np.pi / 3)
#          file, map, colour map, limits, colour-bar label, label size, page width and height, top of the panels, top of the bar
figures = [("berry_grid_1", 0, "bwr", -3.5, 3.5, r"occupied Berry curvature  $\Omega(\mathbf{k})$  ($a^2$)", 13, 1032.59, 283.68, 30.2, 7.0),
           ("metric_trace_grid", 1, "viridis", 0, 4.0, r"occupied quantum metric  $\mathrm{tr}\,g(\mathbf{k})$  ($a^2$)", 13, 1034.11, 296.26, 35.2, 11.5),
           ("metric_det_grid", 2, "viridis", 0, 4.0, r"occupied quantum-metric determinant  $\det g(\mathbf{k})$  ($a^4$)", 9, 1029.11, 296.26, 35.2, 11.5)]

with plt.rc_context(big_style):
    for name, i, cmap, vmin, vmax, title, size, W, H, top, bar_top in figures:
        fig = plt.figure(figsize=(W / 72, H / 72))
        for j, (model, B) in enumerate(cases):
            ax = panel(fig, 7.0 + 240.2 * j, top, 222.6, 222.6)
            im = ax.pcolormesh(k_xv, k_yv, maps[model, B][i], cmap=cmap, vmin=vmin, vmax=vmax, shading="auto", rasterized=True)
            ax.contour(k_xv, k_yv, maps[model, B][i], levels=np.linspace(vmin, vmax, 9), colors="w" if cmap == "viridis" else "0.6",
                       linewidths=0.4, linestyles="dashed", alpha=0.7)
            ax.plot(hex_x, hex_y, "k-", lw=2)
            ax.set_xlim(-L, L)
            ax.set_ylim(-L, L)
            ax.set_xticks([])
            ax.set_yticks([])
            ax.text(0.011, 1 + 13 / 222.6, "(" + "abcd"[j] + ")", transform=ax.transAxes, fontsize=18, fontweight="bold", va="center")
        cb = fig.colorbar(im, cax=panel(fig, 966.8, bar_top, 12.2, 270))
        cb.set_label(title, fontsize=size)
        plt.savefig(f"figures/{name}.pdf")
        plt.show()